# Smog City — Week 5: Data Audit and Quality Check

**Dataset:** `aqi_india_38cols_knn_final.csv` (the file used in Week 4)

## Aim
Audit the dataset before further analysis: inspect its dimensions, data types, missing values, duplicates, descriptive statistics, and AQI distribution.

**How to use:** Open in Google Colab, run cells from top to bottom, and upload the same CSV when prompted. This notebook does not overwrite the source file.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
from IPython.display import display

pd.set_option("display.max_columns", 100)
print("Week 5 — Data Audit")

In [ ]:
uploaded = files.upload()
filename = next(iter(uploaded))
df = pd.read_csv(filename, low_memory=False)
df.columns = df.columns.astype(str).str.strip().str.lower()

print("Loaded file:", filename)
print("Rows, columns:", df.shape)
display(df.head())
print("\nColumns:")
print(df.columns.tolist())

In [ ]:
# 1. Dataset structure and quality
print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))
print("Duplicate rows:", f"{df.duplicated().sum():,}")
print("Total missing cells:", f"{df.isna().sum().sum():,}")

audit = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=True)
}).sort_values("missing_percent", ascending=False)
display(audit)
audit.to_csv("Week5_Data_Audit.csv")
print("Saved: Week5_Data_Audit.csv")

In [ ]:
# 2. Descriptive statistics for numeric columns
numeric_df = df.select_dtypes(include=np.number)
display(numeric_df.describe().T)
numeric_df.describe().T.to_csv("Week5_Descriptive_Statistics.csv")
print("Saved: Week5_Descriptive_Statistics.csv")

In [ ]:
# 3. Missing values chart (show only columns that have missing data)
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]

if len(missing):
    plt.figure(figsize=(12, 5))
    missing.plot(kind="bar")
    plt.title("Week 5 — Missing Values by Column")
    plt.xlabel("Column")
    plt.ylabel("Missing cells")
    plt.xticks(rotation=75, ha="right")
    plt.tight_layout()
    plt.savefig("Week5_Missing_Values.png", dpi=200)
    plt.show()
else:
    print("No missing values found.")


In [ ]:
# 4. AQI distribution (use the AQI column if present)
aqi_candidates = ["us_aqi", "aqi", "india_aqi", "aqi_value"]
aqi_col = next((c for c in aqi_candidates if c in df.columns), None)

if aqi_col:
    aqi_values = pd.to_numeric(df[aqi_col], errors="coerce").dropna()
    plt.figure(figsize=(8, 5))
    plt.hist(aqi_values, bins=30)
    plt.title(f"Week 5 — Distribution of {aqi_col}")
    plt.xlabel(aqi_col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.savefig("Week5_AQI_Distribution.png", dpi=200)
    plt.show()
    print("AQI column used:", aqi_col)
else:
    print("No recognized AQI column found. Available columns:", df.columns.tolist())


In [ ]:
# 5. Export a compact summary
summary = pd.DataFrame({
    "Metric": ["Rows", "Columns", "Duplicate rows", "Missing cells"],
    "Value": [len(df), len(df.columns), int(df.duplicated().sum()), int(df.isna().sum().sum())]
})
display(summary)
summary.to_csv("Week5_Summary.csv", index=False)
print("Week 5 audit complete. Outputs are saved in the Colab session.")

## Week 5 conclusion
After running all cells, write 2–4 sentences describing the actual results: dataset size, number of duplicates, whether missing values remain, and the overall shape of the AQI distribution. Use the printed results rather than assuming the data is perfect.